In [1]:
"""
Customer Churn Analysis - Data Cleaning & Exploratory Data Analysis
Dataset: IBM Telco Customer Churn (7,043 customers, 21 columns)
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

In [3]:
# ---------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------
df = pd.read_csv("data/telco_churn.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies   

In [4]:
# ---------------------------------------------------------
# 2. CLEAN DATA
# ---------------------------------------------------------
# TotalCharges has blank strings for new customers (tenure=0) -> convert to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("\nMissing values before fix:\n", df.isnull().sum()[df.isnull().sum() > 0])


Missing values before fix:
 TotalCharges    11
dtype: int64


In [5]:
# Fill missing TotalCharges with 0 (these are customers with 0 tenure, i.e. brand new)
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [6]:
# Drop customerID (not useful for analysis, just an identifier)
df.drop(columns=["customerID"], inplace=True)

In [7]:
# Standardize SeniorCitizen to Yes/No for readability
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

In [8]:
# Confirm no duplicates
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 22


In [9]:
df.to_csv("data/telco_churn_clean.csv", index=False)
print("\nCleaned data saved to data/telco_churn_clean.csv")


Cleaned data saved to data/telco_churn_clean.csv


In [10]:
# ---------------------------------------------------------
# 3. HIGH-LEVEL CHURN OVERVIEW
# ---------------------------------------------------------
churn_rate = df["Churn"].value_counts(normalize=True) * 100
print("\nChurn rate:\n", churn_rate)


Churn rate:
 Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [11]:
plt.figure(figsize=(5, 5))
colors = ["#2E86AB", "#E63946"]
plt.pie(df["Churn"].value_counts(), labels=["Stayed", "Churned"], autopct="%1.1f%%",
        colors=colors, startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
plt.title("Overall Customer Churn Rate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("images/01_overall_churn_rate.png")
plt.close()

In [12]:
# ---------------------------------------------------------
# 4. CHURN BY CONTRACT TYPE
# ---------------------------------------------------------
plt.figure(figsize=(6, 4.5))
ct = pd.crosstab(df["Contract"], df["Churn"], normalize="index") * 100
ct.plot(kind="bar", stacked=True, color=["#2E86AB", "#E63946"], ax=plt.gca())
plt.title("Churn Rate by Contract Type", fontsize=13, fontweight="bold")
plt.ylabel("Percentage of Customers")
plt.xlabel("Contract Type")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.savefig("images/02_churn_by_contract.png")
plt.close()

In [13]:
# ---------------------------------------------------------
# 5. CHURN BY TENURE (how long they've stayed)
# ---------------------------------------------------------
plt.figure(figsize=(7, 4.5))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack",
             palette=["#2E86AB", "#E63946"], bins=30)
plt.title("Customer Tenure Distribution by Churn Status", fontsize=13, fontweight="bold")
plt.xlabel("Tenure (months)")
plt.tight_layout()
plt.savefig("images/03_tenure_distribution.png")
plt.close()

In [14]:
# ---------------------------------------------------------
# 6. CHURN BY MONTHLY CHARGES
# ---------------------------------------------------------
plt.figure(figsize=(7, 4.5))
sns.kdeplot(data=df[df.Churn == "No"], x="MonthlyCharges", fill=True, color="#2E86AB", label="Stayed", alpha=0.5)
sns.kdeplot(data=df[df.Churn == "Yes"], x="MonthlyCharges", fill=True, color="#E63946", label="Churned", alpha=0.5)
plt.title("Monthly Charges Distribution by Churn Status", fontsize=13, fontweight="bold")
plt.xlabel("Monthly Charges ($)")
plt.legend()
plt.tight_layout()
plt.savefig("images/04_monthly_charges.png")
plt.close()

In [15]:
# ---------------------------------------------------------
# 7. CHURN BY INTERNET SERVICE TYPE
# ---------------------------------------------------------
plt.figure(figsize=(6, 4.5))
ct2 = pd.crosstab(df["InternetService"], df["Churn"], normalize="index") * 100
ct2.plot(kind="bar", stacked=True, color=["#2E86AB", "#E63946"], ax=plt.gca())
plt.title("Churn Rate by Internet Service Type", fontsize=13, fontweight="bold")
plt.ylabel("Percentage of Customers")
plt.xlabel("Internet Service")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.savefig("images/05_churn_by_internet_service.png")
plt.close()

In [16]:
# ---------------------------------------------------------
# 8. CHURN BY PAYMENT METHOD
# ---------------------------------------------------------
plt.figure(figsize=(7, 4.5))
ct3 = pd.crosstab(df["PaymentMethod"], df["Churn"], normalize="index") * 100
ct3.sort_values("Yes", ascending=False).plot(kind="barh", stacked=True,
                                              color=["#2E86AB", "#E63946"], ax=plt.gca())
plt.title("Churn Rate by Payment Method", fontsize=13, fontweight="bold")
plt.xlabel("Percentage of Customers")
plt.legend(title="Churn")
plt.tight_layout()
plt.savefig("images/06_churn_by_payment_method.png")
plt.close()

In [17]:
print("\nAll EDA charts saved to images/ folder.")


All EDA charts saved to images/ folder.


In [18]:
# ---------------------------------------------------------
# 9. KEY NUMBERS FOR THE REPORT / README
# ---------------------------------------------------------
summary = {
    "Total customers": len(df),
    "Churned customers": (df.Churn == "Yes").sum(),
    "Overall churn rate (%)": round(churn_rate["Yes"], 2),
    "Month-to-month churn rate (%)": round(ct.loc["Month-to-month", "Yes"], 2),
    "Two year contract churn rate (%)": round(ct.loc["Two year", "Yes"], 2),
    "Fiber optic churn rate (%)": round(ct2.loc["Fiber optic", "Yes"], 2),
    "Avg tenure - churned (months)": round(df[df.Churn == "Yes"].tenure.mean(), 1),
    "Avg tenure - stayed (months)": round(df[df.Churn == "No"].tenure.mean(), 1),
    "Avg monthly charges - churned ($)": round(df[df.Churn == "Yes"].MonthlyCharges.mean(), 2),
    "Avg monthly charges - stayed ($)": round(df[df.Churn == "No"].MonthlyCharges.mean(), 2),
}
print("\n=== KEY INSIGHTS ===")
for k, v in summary.items():
    print(f"{k}: {v}")


=== KEY INSIGHTS ===
Total customers: 7043
Churned customers: 1869
Overall churn rate (%): 26.54
Month-to-month churn rate (%): 42.71
Two year contract churn rate (%): 2.83
Fiber optic churn rate (%): 41.89
Avg tenure - churned (months): 18.0
Avg tenure - stayed (months): 37.6
Avg monthly charges - churned ($): 74.44
Avg monthly charges - stayed ($): 61.27


In [19]:
with open("key_insights.txt", "w") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

In [20]:
"""
Customer Churn Analysis - Predictive Model
Builds a Logistic Regression + Random Forest model to predict churn
and extracts the top factors driving churn.
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

In [21]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

In [22]:
# ---------------------------------------------------------
# 1. LOAD CLEAN DATA
# ---------------------------------------------------------
df = pd.read_csv("data/telco_churn_clean.csv")

In [23]:
# ---------------------------------------------------------
# 2. ENCODE CATEGORICAL VARIABLES
# ---------------------------------------------------------
df_model = df.copy()
target = df_model.pop("Churn").map({"No": 0, "Yes": 1})

In [24]:
cat_cols = df_model.select_dtypes(include="object").columns.tolist()
df_encoded = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

/tmp/ipykernel_628/2767866570.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_model.select_dtypes(include="object").columns.tolist()


In [25]:
# ---------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df_encoded, target, test_size=0.2, random_state=42, stratify=target
)

In [26]:
scaler = StandardScaler()
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

In [27]:
# ---------------------------------------------------------
# 4. LOGISTIC REGRESSION MODEL
# ---------------------------------------------------------
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

In [28]:
# ---------------------------------------------------------
# 5. RANDOM FOREST MODEL
# ---------------------------------------------------------
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

In [29]:
# ---------------------------------------------------------
# 6. EVALUATE BOTH MODELS
# ---------------------------------------------------------
def evaluate(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 3),
        "Precision": round(precision_score(y_true, y_pred), 3),
        "Recall": round(recall_score(y_true, y_pred), 3),
        "F1-score": round(f1_score(y_true, y_pred), 3),
        "ROC-AUC": round(roc_auc_score(y_true, y_prob), 3),
    }

In [30]:
results = pd.DataFrame([
    evaluate("Logistic Regression", y_test, y_pred_lr, y_prob_lr),
    evaluate("Random Forest", y_test, y_pred_rf, y_prob_rf),
])
print(results.to_string(index=False))
results.to_csv("model_results.csv", index=False)

              Model  Accuracy  Precision  Recall  F1-score  ROC-AUC
Logistic Regression     0.806      0.659   0.559     0.605    0.842
      Random Forest     0.761      0.534   0.783     0.635    0.843


In [31]:
# ---------------------------------------------------------
# 7. CONFUSION MATRIX (Random Forest - best model typically)
# ---------------------------------------------------------
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Stayed", "Churned"], yticklabels=["Stayed", "Churned"])
plt.title("Confusion Matrix - Random Forest", fontsize=13, fontweight="bold")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("images/07_confusion_matrix.png")
plt.close()

In [32]:
# ---------------------------------------------------------
# 8. FEATURE IMPORTANCE (what actually drives churn)
# ---------------------------------------------------------
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(12)

In [33]:
plt.figure(figsize=(7, 5.5))
top_features.sort_values().plot(kind="barh", color="#2E86AB")
plt.title("Top 12 Factors Driving Customer Churn", fontsize=13, fontweight="bold")
plt.xlabel("Feature Importance")
plt.tight_layout()
plt.savefig("images/08_feature_importance.png")
plt.close()

In [34]:
print("\nTop churn drivers:\n", top_features)


Top churn drivers:
 tenure                                  0.180675
TotalCharges                            0.127160
Contract_Two year                       0.124873
MonthlyCharges                          0.080644
InternetService_Fiber optic             0.079952
PaymentMethod_Electronic check          0.060620
Contract_One year                       0.047824
OnlineSecurity_Yes                      0.035111
TechSupport_Yes                         0.024552
TechSupport_No internet service         0.022542
DeviceProtection_No internet service    0.020589
OnlineSecurity_No internet service      0.020043
dtype: float64


In [35]:
with open("model_results.txt", "w") as f:
    f.write(results.to_string(index=False))
    f.write("\n\nTop churn drivers (Random Forest feature importance):\n")
    f.write(top_features.to_string())

In [36]:
print("\nModel evaluation + feature importance charts saved.")


Model evaluation + feature importance charts saved.
